# 18 - CIDER T-cell Score vs ICI Response, by MSI Status

Supplementary figure: does CIDER T-cell Score still separate ICI Responders
from Non-Responders once patients are grouped by microsatellite instability
(MSI) status rather than by cancer type?

Built exactly like Figure 3's main panel (`response_plot()`), but with a
new MSI-status grouping in place of `cohort_broad_go`, at two levels of
granularity:

- **ICI-Treated (MSS)** / **ICI-Treated (MSI)** — every ICI-treated
  patient with known MSI status, split only by MSI-high vs not.
- **CRC-MSI** / **Non-CRC MSI** — the MSI-high patients broken down further:
  `CRC-MSI` is the existing colorectal MSI-high `cohort_broad` category;
  `Non-CRC MSI` pools every other MSI-high patient regardless of cancer type
  (SKCM, NSCLC, BLCA, and Other from the broad cohorts) into a single
  group, since none of those has enough MSI-high ICI-treated patients on
  its own to stand as a separate box.

As with Figure 3's pooled "ICI-Treated" + per-cancer-type boxes, a
patient contributes to **two** boxes here (their ICI-Treated bucket, plus
CRC-MSI/Non-CRC MSI if MSI-high) — this is deliberate, the same
subtotal-plus-detail structure already used throughout the paper, not a
double-counting bug.

**Input:** `cpi_ready.rds`
**Output:** `figs/supplement_msi_response.rds`, `figs/supplement_msi_response.png`


In [ ]:
source(paste0(dirname(getwd()), "/helpers/map.r"))
source(paste0(dirname(getwd()), "/helpers/shortcuts.r"))
source(paste0(dirname(getwd()), "/helpers/config.r"))
source(paste0(dirname(getwd()), "/helpers/themes.r"))

library(ggpubr)
library(patchwork)

dir.create(paste0(CIDER_PROJ_DIR, "figs/"), showWarnings = FALSE, recursive = TRUE)


# 0 - Build MSI cohort grouping

In [ ]:
# ── Load and de-duplicate the pooled "ICI-Treated" copy ───────────────────────
# cpi_ready.rds stores one row per (sampleId, gp) *and* a duplicated copy of
# every row with cohort_broad overwritten to "ICI-Treated" (4_base.ipynb,
# for Figure 3's pooled-vs-per-cancer-type panel). That duplication isn't
# useful here — this figure pools patients by MSI status, not by
# cohort_broad — so it's dropped first to get exactly one row per
# (sampleId, gp) with the patient's real cohort_broad, before building our
# own MSI-based duplication below.
#
# msi_high is numeric 1/0 (as used throughout Table 1 / Table S3); patients
# with unknown MSI status are excluded, since they can't be placed in
# either the MSS or MSI pan-cancer bucket.
base <- readRDS(paste0(CIDER_PROJ_DIR, "cpi_ready.rds")) |>
  filter(cohort_broad != "ICI-Treated", !is.na(msi_high))

# ── ICI-Treated level: every patient, split by MSI status only ─────────────────
pan_cancer <- base |>
  mutate(msi_group = ifelse(msi_high == 1, "ICI-Treated (MSI)", "ICI-Treated (MSS)"))

# ── Sub-cohort level: MSI-high patients only, CRC-MSI vs everything else ──────
# cohort_broad's only categories besides "CRC-MSI" (and the already-dropped
# "ICI-Treated" pool) are SKCM, NSCLC, BLCA, and Other — so "cohort_broad !=
# CRC-MSI" among MSI-high patients is exactly "SKCM, NSCLC, BLCA, or Other",
# pooled into a single "Non-CRC MSI" group.
sub_cohort <- base |>
  filter(msi_high == 1) |>
  mutate(msi_group = ifelse(cohort_broad == "CRC-MSI", "CRC-MSI", "Non-CRC MSI"))

# ── Combine and add sample-size-annotated labels ──────────────────────────────
# Same pattern as cohort_broad_go in 4_base.ipynb: N = number of *unique
# patients* in each msi_group (not rows, since each patient has one row per
# gp), shown in the axis label as "<group> (N = <n>)".
msi_ready <- bind_rows(pan_cancer, sub_cohort) |>
  group_by(msi_group) |>
  mutate(n_msi = n_distinct(sampleId)) |>
  ungroup() |>
  mutate(
    msi_group    = factor(msi_group, levels = MSI_GROUP_LEVELS),
    msi_group_go = {
      lkp <- distinct(pick(msi_group, n_msi)) |> arrange(msi_group)
      factor(
        paste0(msi_group, " (N = ", n_msi, ")"),
        levels = paste0(lkp$msi_group, " (N = ", lkp$n_msi, ")")
      )
    }
  )

message("MSI group patient counts (unique sampleId, any gp):")
msi_ready |>
  distinct(sampleId, msi_group) |>
  count(msi_group, name = "n") |>
  print()

# ── Non-CRC MSI source-cohort breakdown (for the figure caption) ────────────────
# "Non-CRC MSI" pools MSI-high patients from four different Figure 3 broad
# cohorts (SKCM, NSCLC, BLCA, Other) into one box — cohort_broad (the
# original, pre-MSI-grouping label) is still available on sub_cohort, so we
# can report exactly how many patients came from each one.
non_crc_msi_breakdown <- sub_cohort |>
  filter(msi_group == "Non-CRC MSI") |>
  distinct(sampleId, cohort_broad) |>
  count(cohort_broad, name = "n") |>
  arrange(desc(n))

non_crc_msi_caption <- paste0(
  "Non-CRC MSI = ",
  paste0(non_crc_msi_breakdown$cohort_broad, " (n = ", non_crc_msi_breakdown$n, ")", collapse = ", ")
)

message("Non-CRC MSI source-cohort breakdown: ", non_crc_msi_caption)


# 1 - Figure

In [ ]:
# x_dodge = 2 staggers the x-axis labels across two rows — the MSI group
# names ("ICI-Treated (MSS) (N = ...)", etc.) are much longer than the
# short cohort_broad_go labels response_plot() is usually called with, and
# would otherwise overlap on one row.
#
# The caption reports Non-CRC MSI's source-cohort breakdown (computed in the
# previous cell), since that box pools four different Figure 3 broad
# cohorts and the count alone doesn't show that composition.
msi_response_figure <- response_plot(
  msi_ready, "T-cell", "msi_group_go",
  title  = "CIDER T-cell Score vs ICI Response, by MSI Status",
  y_col  = "cider_score_depth_adj",
  x_dodge = 2
) +
  ggplot2::labs(caption = non_crc_msi_caption)

set_plot_dims(9, 4.5); msi_response_figure


In [ ]:
saveRDS(msi_response_figure, paste0(CIDER_PROJ_DIR, "figs/supplement_msi_response.rds"))
ggsave(paste0(CIDER_PROJ_DIR, "figs/supplement_msi_response.png"),
       msi_response_figure, width = 9, height = 4.5, dpi = 300)
message("Saved: supplement_msi_response.rds and supplement_msi_response.png")
